In [97]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.2.2
tiktoken version: 0.12.0


# Load the file we'll tokenize

* It's a public domain short story, [The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict)

In [98]:
# Ensure the file exists locally

import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

print("The book should now exist on your local disk!")

The book should now exist on your local disk!


In [99]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters: ", len(raw_text))
print(raw_text[:99])

Total number of characters:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## Version 1 - Very simple, regex based tokenizer

### Step 1 - Tokenize the text

In [100]:
# Simply split based on spaces. Use a toy-example for now

import re

text="Hello, reader. This, is a test of tokenization."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'reader.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test', ' ', 'of', ' ', 'tokenization.']


In [101]:
# Also split based on commas, periods

result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'reader', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', ' ', 'of', ' ', 'tokenization', '.', '']


In [102]:
# Remove the empty strings

result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'reader', '.', 'This', ',', 'is', 'a', 'test', 'of', 'tokenization', '.']


In [103]:
# Handle even more types of punctuations

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'reader', '.', 'This', ',', 'is', 'a', 'test', 'of', 'tokenization', '.']


In [104]:
# Split the actual book text

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [105]:
# Print the number of tokens

print(len(preprocessed))

4690


### Step 2: Convert tokens into token IDs

In [106]:
# Build a vocabulary using all the unique tokens

all_tokens = sorted(set(preprocessed))
vocab_size = len(all_tokens)
print(vocab_size)

1130


In [107]:
vocab = {token:integer for integer, token in enumerate(all_words)}

for i, item in list(enumerate(vocab.items()))[:50]:
    print(item)

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)


In [108]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.vocab = vocab # Maps strings to integers
        self.vocab_reverse = {i:s for s,i in vocab.items()}

    def encode(self, text):
        # Split into tokens
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [token for token in tokens if token.strip()]

        # Find the corresponding token IDs
        ids = [self.vocab[token] for token in tokens]
        return ids
        
    def decode(self, ids):
        # Find the corresponding strings
        tokens = [self.vocab_reverse[id] for id in ids]

        # Concatenate them
        text = " ".join(tokens)

        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

In [109]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [110]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [111]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

### Step 3: Handle unknown tokens and multiple texts

In [112]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [113]:
len(vocab.items())

1132

In [114]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [121]:
# Update the tokenizer to use these new tokens

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.vocab = vocab
        self.vocab_reverse = {integer:string for string, integer in vocab.items()}

    def encode(self, text):
         # Split into tokens
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [token for token in tokens if token.strip()]

        # Replace unknown tokens appropriately
        tokens = [
            token if token in self.vocab
            else "<|unk|>" for token in tokens
        ]      

        # Find the corresponding token IDs
        ids = [self.vocab[token] for token in tokens]
        return ids

    def decode(self, ids):
        # Find the corresponding strings
        tokens = [self.vocab_reverse[id] for id in ids]

        # Concatenate them
        text = " ".join(tokens)

        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

In [122]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [123]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [118]:
tokenizer.decode(tokenizer.encode(text))

KeyError: 1131